# Multi-Turn Conversations — Claude Managed Agents (CLI)

The same multi-turn flow as [`02-multi-turn/`](../02-multi-turn/), driven from the terminal with Anthropic's **`ant`** CLI instead of the Python SDK.

The headline idea is identical to the SDK version: **create one session and reuse it across turns.** Conversation history lives server-side, so each new message automatically sees everything that came before — you never resend prior turns. The only new mechanics here are shell-flavored:

- Agents and environments are defined as version-controlled `*.yaml` (control plane).
- Each turn repeats the **stream-first** dance — open the stream *before* sending — wrapped in a reusable `chat.sh` helper so the turn cells stay one line each.

> Prerequisite: the `ant` CLI installed and authenticated. See [`01-basics-cli/`](../01-basics-cli/) for install/auth details.

## 1. Setup

Confirm the CLI is installed and can see a credential. The CLI resolves credentials the same way the SDKs do — easiest is `ANTHROPIC_API_KEY` exported from the shared root `.env`.

In [ ]:
%%bash
# If `ant` isn't installed yet (see 01-basics-cli for options):
#   macOS: brew install anthropics/tap/ant
ant --version
# Shows which credential source and workspace the CLI will use (status only).
ant auth status

## 2. Create an Environment

The environment is the **sandboxed container** where tools run. Create it once; in production you'd persist the id and reuse it.

In [ ]:
%%writefile env.yaml
name: multiturn-env-cli
config:
  type: cloud
  networking:
    type: unrestricted

In [ ]:
# Pipe the YAML in via stdin; capture the new environment's id.
# --transform id -r extracts just the id field as a bare string (no quotes).
_env = !ant beta:environments create --transform id -r < env.yaml
env_id = _env[0]
print("Environment ID:", env_id)

## 3. Create an Agent

The agent is a **persisted, versioned config**. The system prompt nudges it to build on the conversation — exactly what multi-turn is about.

In [ ]:
%%writefile agent.yaml
name: Multi-Turn Agent (CLI)
model: claude-opus-4-7
system: |
  You are a helpful assistant. Keep your answers concise and build on the
  conversation so far.
tools:
  - type: agent_toolset_20260401
    default_config:
      enabled: true

In [ ]:
import json

# Capture the full JSON response so we grab BOTH id and version for pinning.
_agent = !ant beta:agents create --format json < agent.yaml
agent = json.loads("".join(_agent))
agent_id, agent_version = agent["id"], agent["version"]
print("Agent ID :", agent_id)
print("Version  :", agent_version)

## 4. Create ONE Session — Reused Across Every Turn

This is the crux of multi-turn. In module 01 the session was per-run. Here we create **a single session** and send it multiple messages. The server keeps the history, so turn 2 already knows what happened in turn 1.

We export the session id as `$SID` so the `%%bash` turn cells below can read it.

In [ ]:
import os

# Pin the agent to its exact version: {"type": "agent", "id": ..., "version": ...}
agent_ref = json.dumps({"type": "agent", "id": agent_id, "version": agent_version})

_session = !ant beta:sessions create --agent '{agent_ref}' --environment-id {env_id} --title "Multi-turn CLI session" --transform id -r
session_id = _session[0]

os.environ["SID"] = session_id  # exported to every bash cell below
print("Session ID:", session_id)

## 5. A Reusable `chat.sh` Helper

Every turn repeats the same stream-first choreography: open the event stream on a file descriptor, send the user message (which triggers the agent loop), then read events until the session goes idle. We factor that into a `chat()` shell function written to `chat.sh`, so each turn cell is a single call.

Because each `%%bash` cell is a fresh shell, every turn cell `source`s `chat.sh` first — but `$SID` persists because we exported it from Python above.

In [ ]:
%%writefile chat.sh
# chat.sh — send one message to $SID and stream the reply (stream-first).
# Usage:  source chat.sh; chat "your message"
chat() {
  set -uo pipefail
  local message="$1"

  # 1. Open the event stream FIRST (stream-before-send), on fd $stream.
  exec {stream}< <(ant beta:sessions:events stream --session-id "$SID" \
    --transform '{type, text: content.#(type=="text").text, err: error.message}' \
    --format yaml)

  # 2. Send the user message — this triggers the agent loop.
  #    Note: keep messages to plain text; they are inlined into YAML here.
  ant beta:sessions:events send --session-id "$SID" >/dev/null <<YAML
events:
  - type: user.message
    content:
      - type: text
        text: $message
YAML

  # 3. Read events until the session goes idle or ends.
  printf 'Agent: '
  local type=
  while IFS= read -r -u "$stream" line; do
    case "$line" in
      "type: session.status_idle")       break ;;   # agent finished its turn
      "type: session.status_terminated") break ;;   # session ended
      "type: session.error")
        IFS= read -r -u "$stream" next || next=
        case "$next" in "err: "*) msg=${next#err: } ;; *) msg=unknown ;; esac
        printf '\n[Error: %s]\n' "$msg"; break ;;
      "type: "*) type=${line#type: } ;;
      "text: "*)
        [ "$type" = agent.message ] || continue
        val=${line#text: }
        case "$val" in "|-"|"|") ;; *) printf '%s' "$val" ;; esac ;;
      "  "*)
        [ "$type" = agent.message ] && printf '%s\n' "${line#  }" ;;
    esac
  done
  exec {stream}<&-   # close the stream
  printf '\n'
}

## 6. Turn 1 — Start the Conversation

Ask a plain question. Nothing special yet — but note we never mention the session history; the server tracks it.

In [ ]:
%%bash
source chat.sh
chat "What's the largest planet in our solar system?"


## 7. Turn 2 — Context Is Preserved

The follow-up says **"it"** and **"they"** — it only makes sense if the agent remembers turn 1. Same session, no history resent.

In [ ]:
%%bash
source chat.sh
chat "How many moons does it have?"


## 8. Turn 3 — Build Deeper

One more follow-up that leans entirely on the accumulated context.

In [ ]:
%%bash
source chat.sh
chat "Name the three largest ones and one interesting fact about each."


## Summary

You just ran a full multi-turn conversation from the terminal:

```
ant beta:environments create   →  one environment (reused)
ant beta:agents create          →  one versioned agent (reused)
ant beta:sessions create        →  ONE session, reused across turns
per turn:  stream FIRST → send → read until idle   (chat.sh)
```

The key difference from module 01: **the session is created once and reused.** Conversation history is maintained server-side, so each `chat` call sees every earlier turn without resending anything.

### CLI vs SDK, same lesson

| | CLI (this module) | SDK ([`02-multi-turn/`](../02-multi-turn/)) |
|---|---|---|
| Reusable turn helper | `chat.sh` shell function | `chat()` Python function |
| Session lifetime | one `$SID`, many sends | one `session.id`, many sends |
| Stream-first | `exec {stream}<` per turn | `with ...events.stream()` per turn |
| Idle check | `session.status_idle` | `session.status_idle` |

### Next

Continue to the next module for tool use — giving the agent real capabilities (bash, files, code execution) inside its environment.